## Xavier Initialization

Xavier initialization (also known as Glorot initialization) in PyTorch is a technique designed to keep the variance of activations and gradients stable across all layers, preventing vanishing or exploding gradients.  It is most effective for networks using sigmoid or tanh activation functions, which are approximately linear around zero

In [12]:
# =============================================================================
#    PINN overview:
#    Input  : [t, ax, ay, az, gx, gy, gz]   (time + 6-axis IMU)
#    Output : [v, θ, μ_eff, x_b, y_b, z_b]  (velocity, tilt, friction,
#                                              brain displacement 3-axis)
#    Derived: P_skid, HIC15, BrIC, Injury_Score
#
#    Physics losses enforced:
#    L_vehicle  — tire-road friction ODE (longitudinal + lateral dynamics)
#    L_biomech  — Kelvin-Voigt brain spring-mass-damper (3-axis)
#    L_data     — MSE fit to IMU sensor readings
#    L_boundary — initial conditions (helmet at rest at t=0)
# =============================================================================


In [13]:
import torch
import matplotlib.pyplot as plt
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from typing import Dict

import numpy as np
import matplotlib.gridspec as gridspec

import pandas as pd
import warnings

warnings.filterwarnings("ignore")

plt.style.use("dark_background")
DARK_ACCENT = "#00FFFF"
WARN_ACCENT = "#FFD600"
CRIT_ACCENT = "#FF1744"

In [14]:
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# ============ DEVICE ==========================================================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[RoadSOS] Running on {DEVICE}")

# ============== Physical Constraints ==========================================

MASS_RIDER = 75.0
MASS_BIKE = 150.0

M = MASS_RIDER + MASS_BIKE
G = 9.81
WHEELBASE = 1.35
MU_DRY = 0.75
MU_WET = 0.45
MU_OIL = 0.15


# --- HIC thresholds (ECE 22.06 standard) ---
HIC_LOW       = 700     # below: low risk
HIC_HIGH      = 1000    # above: severe / likely fatal


# --- BrIC critical angular velocities (rad/s) — NHTSA cadaver data ---
BRIC_THRESH_X = 66.3    # sagittal (pitch)
BRIC_THRESH_Y = 56.5    # coronal  (roll)
BRIC_THRESH_Z = 42.2    # transverse (yaw)
BRIC_CRITICAL = 1.0     # BrIC >= 1 → critical brain injury risk


# --- Kelvin-Voigt brain biomechanics (3-axis) ---
# Brain modelled as viscoelastic mass on spring-damper inside skull
BRAIN_MASS    = 1.4     # kg  (avg human brain)
BRAIN_K       = 2.1e4   # N/m spring stiffness (from literature)
BRAIN_C       = 85.0    # N·s/m damping coefficient
# Natural frequency ≈ sqrt(k/m) / (2π) ≈ 19.5 Hz — in concussion range

[RoadSOS] Running on cuda


Inputs (7): t (time) and 6-axis IMU data `(ax, ay, az, gx, gy, gz)`.
Outputs (6): Physical quantities predicted by the network `(v for velocity, theta for tilt, mu_eff for effective friction, and x_b, y_b, z_b` for brain displacement).

In [15]:
# ── Section 3: PINN Architecture ─────────────────────────────────────────────


class RoadSOSPINN(nn.Module):
  def __init__(
      self,
      input_dim: int = 7,
      output_dim: int = 6,
      hidden_layers: int = 6,
      hidden_width: int = 128,

  ):

      super().__init__()
      self.output_dim = output_dim
      self.input_dim = input_dim # Added to make it accessible
      self.hidden_layers = hidden_layers # Added to make it accessible
      self.hidden_width = hidden_width # Added to make it accessible

      # ── Build layers ────────────────────────────────────────────────────
      layers = []
      layers.append(nn.Linear(input_dim, hidden_width))
      layers.append(nn.Tanh())

      # Hidden → hidden  (with residual-style skip every 2 blocks)
      for i in range(hidden_layers - 1):
          layers.append(nn.Linear(hidden_width, hidden_width))
          layers.append(nn.Tanh())

      # Last Hidden -> Output
      layers.append(nn.Linear(hidden_width, output_dim))

      self.net = nn.Sequential(*layers)

      # ── Weight initialisation: Xavier uniform ────────────────────────
      # Xavier keeps gradient variance stable through tanh activations
      self._init_weights()

  def _init_weights(self):
    for layer in self.net:
      if isinstance(layer, nn.Linear):
        nn.init.xavier_uniform_(layer.weight)
        nn.init.zeros_(layer.bias)

  def forward(self, x: torch.Tensor) -> torch.Tensor:
    return self.net(x)

  def predict_with_derived(
      self, x: torch.Tensor
  ) -> Dict[str, torch.Tensor]:
      out = self.forward(x)

      v       = out[:, 0:1]   # velocity         (m/s,  normalised)
      theta   = out[:, 1:2]   # tilt angle        (rad)
      mu_eff  = out[:, 2:3]   # effective friction (0–1)
      x_b     = out[:, 3:4]   # brain disp x      (mm)
      y_b     = out[:, 4:5]   # brain disp y      (mm)
      z_b     = out[:, 5:6]   # brain disp z      (mm)

      # P_skid: sigmoid of how far friction is below safe threshold
      # When mu_eff << MU_WET, skid is imminent
      mu_safe = torch.tensor(MU_WET, device=x.device)
      P_skid  = torch.sigmoid(10.0 * (mu_safe - mu_eff))

      # BrIC from peak angular velocities in input
      omega_x = x[:, 4:5]
      omega_y = x[:, 5:6]
      omega_z = x[:, 6:7]
      BrIC = torch.sqrt(
          (omega_x / BRIC_THRESH_X) ** 2 +
          (omega_y / BRIC_THRESH_Y) ** 2 +
          (omega_z / BRIC_THRESH_Z) ** 2
      )

      # Brain tissue acceleration (second derivative of displacement)
      # Approximated here via KV model: a_brain = (k·x_b + c·v_b) / m
      # At inference we use the raw displacement as a proxy for injury
      brain_disp_mag = torch.sqrt(x_b**2 + y_b**2 + z_b**2)

      return {
          "v":              v,
          "theta":          theta,
          "mu_eff":         mu_eff,
          "x_brain":        x_b,
          "y_brain":        y_b,
          "z_brain":        z_b,
          "P_skid":         P_skid,
          "BrIC":           BrIC,
          "brain_disp_mag": brain_disp_mag,
      }

In [16]:
# Note: The definition for IMUNormaliser class was not provided in the notebook context,
# so it cannot be restored. Proceeding with model instantiation and a basic sanity check.

model = RoadSOSPINN().to(DEVICE)

# Basic Sanity Check
print(f"Model instantiated successfully: {model}")
dummy_input = torch.randn(1, model.input_dim).to(DEVICE) # Use model's input_dim
dummy_output = model(dummy_input)
print(f"Dummy input shape: {dummy_input.shape}")
print(f"Dummy output shape: {dummy_output.shape}")

Model instantiated successfully: RoadSOSPINN(
  (net): Sequential(
    (0): Linear(in_features=7, out_features=128, bias=True)
    (1): Tanh()
    (2): Linear(in_features=128, out_features=128, bias=True)
    (3): Tanh()
    (4): Linear(in_features=128, out_features=128, bias=True)
    (5): Tanh()
    (6): Linear(in_features=128, out_features=128, bias=True)
    (7): Tanh()
    (8): Linear(in_features=128, out_features=128, bias=True)
    (9): Tanh()
    (10): Linear(in_features=128, out_features=128, bias=True)
    (11): Tanh()
    (12): Linear(in_features=128, out_features=6, bias=True)
  )
)
Dummy input shape: torch.Size([1, 7])
Dummy output shape: torch.Size([1, 6])


# GREAT SUCCESS